# 03 — ML Model Training with Lance DataLoader

**Purpose:** Demonstrate end-to-end model training using the Lance `frames` dataset as the data source, showcasing random-access batch sampling and version pinning for reproducibility. Includes a throughput benchmark against an equivalent Delta table.

**What this notebook does:**

1. **Pin the training dataset version.** Open the Lance `frames` dataset at a specific version (e.g., v2, which includes CLIP embeddings). Pinning a version is the Lance equivalent of a data snapshot — training runs that reference the same version number are guaranteed to see the same data, regardless of subsequent appends or FE updates.

2. **Configure the Lance DataLoader.** Build a `lance.torch.data.LanceDataset` with:
   - **Column projection:** load only `image` and `embedding` (or `label` if present), skipping unused columns to minimize I/O per batch.
   - **Random sampling:** Lance serves each batch via direct row-index seeks — no shuffle buffer, no epoch-level sort. This is the key difference from Delta/Parquet, where random access requires reading entire row groups.
   - **Batch size and prefetch:** tune for GPU utilization.

3. **Define the model and training loop.** Use frozen CLIP embeddings as input features and fine-tune a lightweight classification head on top. The training loop is intentionally simple — the focus of this notebook is data loading, not model architecture.

4. **Lance vs Delta throughput benchmark.** Before training, run a controlled benchmark:
   - Lance: time N random-access batches using `LanceDataset`.
   - Delta: write an equivalent dataset to a Delta table, shuffle with `orderBy(rand())`, read N batches via Spark → Pandas → Tensor.
   - Report samples/second for both. The gap widens as dataset size grows; at 10M+ rows the Delta shuffle becomes a distributed sort that dominates epoch time.

5. **Train and log to MLflow.** Run the training loop and log loss, accuracy, and data loading throughput to an MLflow experiment. Pin the Lance dataset version as a parameter so the run is fully reproducible.

**Inputs:** Lance `frames` dataset v2 at `/Volumes/{catalog}/{schema}/{volume}/frames/` (with embeddings)

**Outputs:** Trained model checkpoint logged to MLflow; throughput benchmark results.